# Use PBI package

In [ ]:
# In notebook
import pbi
import time

# Initialize (returns immediately!)
start = time.time()
retriever = pbi.quick_connect()
print(f"✅ Initialized in {time.time() - start:.2f}s")

# Do other work while FASTA loads in background
print("🔄 FASTA files loading in background...")
print("You can run queries immediately - they'll wait for loading if needed")

# Check if ready
if retriever.is_ready():
    print("✅ FASTA files loaded!")
else:
    print("⏳ Still loading...")

# This will wait for loading to complete if needed
stats = retriever.get_stats()

2025-11-24 17:54:35,793 - INFO - 📂 Checking FASTA index files:
2025-11-24 17:54:35,794 - INFO -    Phage index: True (52570.4 KB)
2025-11-24 17:54:35,794 - INFO -    Protein index: True (1432185.2 KB)
2025-11-24 17:54:35,795 - INFO - 📂 Connecting to database: /home/twg/workplace/PBI/data/processed/databases/phage_database_optimized.duckdb
2025-11-24 17:54:35,824 - INFO - 🔄 Starting background FASTA loading...
2025-11-24 17:54:35,825 - INFO - 🔄 [Background] Loading phage FASTA: /home/twg/workplace/PBI/data/processed/sequences/all_phages.fasta
2025-11-24 17:54:35,825 - INFO - ✅ Initialization complete (FASTA loading in background)
2025-11-24 17:54:35,828 - INFO - ⏳ Waiting for FASTA loading to complete...


✅ Initialized in 0.03s
🔄 FASTA files loading in background...
You can run queries immediately - they'll wait for loading if needed
⏳ Still loading...


2025-11-24 17:57:17,915 - INFO -    ✅ Phage FASTA loaded in 162.09s (873,717 sequences)
2025-11-24 17:57:17,916 - INFO - 🔄 [Background] Loading protein FASTA: /home/twg/workplace/PBI/data/processed/sequences/all_proteins.fasta
2025-11-24 17:59:40,889 - WARNING - ⚠️ Timeout after 300s - FASTA may still be loading


In [ ]:
print(f"📊 Database stats: {stats}")

In [ ]:
# Initialize with background loading
retriever = pbi.quick_connect()

# Do database queries while FASTA loads
db_stats = retriever.conn.execute("SELECT COUNT(*) FROM fact_phages").fetchone()[0]
print(f"Database has {db_stats:,} phages")

# Now wait for FASTA to be ready
retriever.wait_until_ready(timeout=300)  # Wait up to 5 minutes

# Now use sequences
df = retriever.get_phage_sequences("SELECT Phage_ID FROM fact_phages LIMIT 10")

In [ ]:
retriever.help()  # Display help information


        SequenceRetriever Help:
        
        Methods:
            - get_phage_sequences(query: str, limit: Optional[int] = None) -> pd.DataFrame
            - get_protein_sequences(query: str, limit: Optional[int] = None) -> pd.DataFrame
            - get_sequences_by_ids(phage_ids: Optional[List[str]] = None, protein_ids: Optional[List[str]] = None) -> Dict
            - get_protein_sequences_by_phage(phage_id: str) -> pd.DataFrame
            - export_fasta(df: pd.DataFrame, output_path: str, id_col: str = 'Phage_ID')
            - get_stats() -> Dict
            - close()
        
        Usage Examples:
            retriever = SequenceRetriever(db_path, phage_fasta_path, protein_fasta_path)
            phage_df = retriever.get_phage_sequences("SELECT Phage_ID FROM fact_phages WHERE Length > 50000", limit=100)
            protein_df = retriever.get_protein_sequences("SELECT Protein_ID FROM dim_proteins WHERE Molecular_weight > 50000", limit=100)
            retriever.export_fas